# Phase 4b — Open-Set: Comparing Distance/Scoring Methods
Naive cosine/Euclidean distance-to-prototype gave AUROC ~ chance (0.44-0.59) and Unknown-F1 = 0.0 -- the model never flagged Stage 4/5 as unknown.
This notebook tests whether a smarter distance metric fixes that, using the identical unknown-class setup, splits, and seeds as Phase 4.

**Methods compared:**
1. Cosine distance to nearest prototype (Phase 4 baseline, rerun here for a single side-by-side table)
2. Euclidean distance to nearest prototype (Phase 4 baseline, rerun here)
3. **Mahalanobis distance** to nearest prototype -- accounts for how much each known class's embeddings naturally vary, so "far in a direction the class doesn't normally spread" counts as more unusual than "far in a direction it does"

**Unknown classes (same as Phase 4):** Stage 4 ROP, Stage 5 ROP -- excluded from all prototype/covariance construction, used only as test-time rejection targets.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import f1_score, roc_auc_score

SHOTS = [1, 3, 5]
SEEDS = [0, 1, 2, 3, 4]
UNKNOWN_CLASSES = ["Stage 4 ROP", "Stage 5 ROP"]

EMB_PATH = r"E:\rop\codes\eyeclip_embeddings.zip"
FUSED_EMB_PATH = r"E:\rop\codes\fused_embeddings.npz"
SPLIT_DIR = Path(r"E:\rop\dataset\ROP-VL\split")  # CHANGE to actual path

data = np.load(EMB_PATH)
X_train_img, X_val_img, X_test_img = data["X_train"], data["X_val"], data["X_test"]

fused = np.load(FUSED_EMB_PATH)
F_train, F_val, F_test = fused["F_train"], fused["F_val"], fused["F_test"]

train_df = pd.read_csv(SPLIT_DIR / "train.csv")
val_df   = pd.read_csv(SPLIT_DIR / "val.csv")
test_df  = pd.read_csv(SPLIT_DIR / "test.csv")

for name, arr, df in [("X_train", X_train_img, train_df), ("X_val", X_val_img, val_df),
                       ("X_test", X_test_img, test_df), ("F_train", F_train, train_df),
                       ("F_val", F_val, val_df), ("F_test", F_test, test_df)]:
    assert len(arr) == len(df), f"{name} row count ({len(arr)}) != matching split csv ({len(df)})"

all_classes = sorted(train_df["categories"].unique())
known_classes = [c for c in all_classes if c not in UNKNOWN_CLASSES]

y_train_str = train_df["categories"].values
y_val_str   = val_df["categories"].values
y_test_str  = test_df["categories"].values

train_known_mask = np.isin(y_train_str, known_classes)
val_known_mask   = np.isin(y_val_str, known_classes)
test_known_mask  = np.isin(y_test_str, known_classes)

print("known classes:", known_classes)
print("unknown classes:", UNKNOWN_CLASSES)

known classes: ['A-ROP', 'Laser-treated ROP', 'Stage 1 ROP', 'Stage 2 ROP', 'Stage 3 ROP', 'normal']
unknown classes: ['Stage 4 ROP', 'Stage 5 ROP']


## Prototype + distance functions (cosine, euclidean, mahalanobis)

**Mahalanobis, explained simply:** instead of measuring raw straight-line distance, it first learns "how spread out" the known-class embeddings are overall (a shared covariance matrix), then measures distance in units of that natural spread. A test point that's far from every prototype *in a direction none of the known classes vary in* gets a high Mahalanobis distance, even if its raw Euclidean distance looks moderate -- this is what should make unknowns stand out better.

In [2]:
def build_prototypes(X, y, classes, k, rng):
    """Sample k examples per class and average. Identical to Phase 2/3/4."""
    prototypes = []
    for c in classes:
        idx = np.where(y == c)[0]
        n_take = min(k, len(idx))
        chosen = rng.choice(idx, size=n_take, replace=False)
        prototypes.append(X[chosen].mean(axis=0))
    return np.stack(prototypes)


def fit_shared_covariance(X_known, y_known, known_classes, eps=1e-3):
    """Fit one shared covariance matrix across all known-class training embeddings
    (standard 'tied covariance' Mahalanobis setup -- avoids needing enough samples
    per class to invert a per-class covariance matrix, which small classes don't have).
    Input:  X_known [N,D], y_known [N] string labels
    Output: inverse covariance matrix [D,D]
    """
    centered = []
    for c in known_classes:
        Xc = X_known[y_known == c]
        if len(Xc) > 1:
            centered.append(Xc - Xc.mean(axis=0))
    centered = np.concatenate(centered, axis=0)
    cov = np.cov(centered, rowvar=False)
    cov += eps * np.eye(cov.shape[0])  # regularize -- raw cov is likely singular/ill-conditioned
    return np.linalg.inv(cov)


def compute_distances(X, prototypes, metric, inv_cov=None):
    """Return [N, C] distance matrix, one column per prototype. Larger = farther."""
    if metric == "cosine":
        X_n = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)
        P_n = prototypes / (np.linalg.norm(prototypes, axis=1, keepdims=True) + 1e-8)
        return 1.0 - (X_n @ P_n.T)
    elif metric == "euclidean":
        return np.linalg.norm(X[:, None, :] - prototypes[None, :, :], axis=2)
    elif metric == "mahalanobis":
        assert inv_cov is not None, "mahalanobis requires inv_cov"
        dists = np.zeros((X.shape[0], prototypes.shape[0]))
        for c_idx, proto in enumerate(prototypes):
            diff = X - proto[None, :]
            dists[:, c_idx] = np.sqrt(np.einsum("ij,jk,ik->i", diff, inv_cov, diff))
        return dists
    else:
        raise ValueError(f"unknown metric: {metric}")


def nearest_and_min_dist(dist_matrix, classes):
    nearest_idx = dist_matrix.argmin(axis=1)
    min_dist = dist_matrix.min(axis=1)
    preds = np.array([classes[i] for i in nearest_idx])
    return min_dist, preds

In [3]:
def select_threshold(val_dist, val_is_known, n_candidates=50):
    """Identical to Phase 4 -- sweep thresholds on val, pick best known-vs-unknown F1."""
    candidates = np.linspace(val_dist.min(), val_dist.max(), n_candidates)
    best_thresh, best_f1 = candidates[0], -1.0
    for t in candidates:
        pred_known = val_dist <= t
        f1 = f1_score(val_is_known, pred_known, zero_division=0)
        if f1 > best_f1:
            best_f1, best_thresh = f1, t
    return best_thresh


def evaluate_open_set(X_train_known, y_train_known, X_val, val_is_known,
                       X_test, test_is_known, y_test_str, known_classes,
                       k, seed, metric, inv_cov=None):
    rng = np.random.default_rng(seed)
    prototypes = build_prototypes(X_train_known, y_train_known, known_classes, k, rng)

    val_dm  = compute_distances(X_val,  prototypes, metric, inv_cov)
    test_dm = compute_distances(X_test, prototypes, metric, inv_cov)
    val_dist, _            = nearest_and_min_dist(val_dm, known_classes)
    test_dist, test_preds  = nearest_and_min_dist(test_dm, known_classes)

    threshold = select_threshold(val_dist, val_is_known)
    test_accepted = test_dist <= threshold

    unknown_f1 = f1_score(~test_is_known, ~test_accepted, zero_division=0)

    eval_mask = test_is_known & test_accepted
    known_f1 = (f1_score(y_test_str[eval_mask], test_preds[eval_mask],
                          labels=known_classes, average="macro", zero_division=0)
                if eval_mask.sum() > 0 else 0.0)

    try:
        auroc = roc_auc_score(~test_is_known, test_dist)
    except ValueError:
        auroc = float("nan")

    return {"threshold": threshold, "known_f1": known_f1, "unknown_f1": unknown_f1, "auroc": auroc}

## Run all combinations: {image-only, fused} x {cosine, euclidean, mahalanobis} x {1,3,5-shot} x 5 seeds
Mahalanobis's covariance is fit once per representation (image-only vs fused) from the full known-class train set -- not re-fit per shot, since covariance estimation needs more data than a k-shot support set provides.

In [4]:
representations = {
    "image_only": (X_train_img[train_known_mask], y_train_str[train_known_mask], X_val_img, X_test_img),
    "image_plus_metadata": (F_train[train_known_mask], y_train_str[train_known_mask], F_val, F_test),
}

all_results = []

for repr_name, (X_tr_known, y_tr_known, X_va, X_te) in representations.items():
    inv_cov = fit_shared_covariance(X_tr_known, y_tr_known, known_classes)

    for metric in ["cosine", "euclidean", "mahalanobis"]:
        for k in SHOTS:
            for seed in SEEDS:
                res = evaluate_open_set(
                    X_tr_known, y_tr_known, X_va, val_known_mask,
                    X_te, test_known_mask, y_test_str, known_classes,
                    k, seed, metric, inv_cov=inv_cov if metric == "mahalanobis" else None)
                res.update({"representation": repr_name, "metric": metric, "shot": k, "seed": seed})
                all_results.append(res)
    print(f"done: {repr_name}")

all_results_df = pd.DataFrame(all_results)

done: image_only
done: image_plus_metadata


In [5]:
summary = all_results_df.groupby(["representation", "metric", "shot"]).agg(
    known_f1_mean=("known_f1", "mean"), known_f1_std=("known_f1", "std"),
    unknown_f1_mean=("unknown_f1", "mean"), unknown_f1_std=("unknown_f1", "std"),
    auroc_mean=("auroc", "mean"), auroc_std=("auroc", "std"),
).reset_index()

pd.set_option("display.width", 150)
print(summary.to_string(index=False))
summary.to_csv("phase4b_distance_method_comparison.csv", index=False)

     representation      metric  shot  known_f1_mean  known_f1_std  unknown_f1_mean  unknown_f1_std  auroc_mean  auroc_std
         image_only      cosine     1       0.159462      0.019955         0.000000        0.000000    0.487148   0.040215
         image_only      cosine     3       0.147513      0.018189         0.000000        0.000000    0.444030   0.055018
         image_only      cosine     5       0.150560      0.015205         0.000000        0.000000    0.502466   0.016532
         image_only   euclidean     1       0.157980      0.021535         0.000000        0.000000    0.483362   0.040664
         image_only   euclidean     3       0.148092      0.015630         0.000000        0.000000    0.441778   0.054825
         image_only   euclidean     5       0.148813      0.013297         0.000000        0.000000    0.500757   0.016310
         image_only mahalanobis     1       0.161514      0.033764         0.000000        0.000000    0.530344   0.058168
         image_o

## Interpretation notes
- Look at `auroc_mean` first: if Mahalanobis rows show AUROC clearly above 0.5-0.6 while cosine/euclidean stay near chance, that confirms Mahalanobis captures separability the simpler metrics miss.
- If Mahalanobis *also* stays near chance, the problem isn't the distance formula -- it's that the underlying frozen embeddings don't encode enough information to distinguish Stage 4/5 from the 6 known classes at all. That's a deeper limitation (worth stating honestly), not something a smarter distance metric can fix.
- Compare `unknown_f1_mean` the same way: still 0.0 across the board would mean thresholding never accepts a meaningful reject region even when AUROC improves -- check `threshold_mean`/`known_f1_mean` trade-off in the raw `all_results_df` if that happens.
- Whatever the outcome, report all three methods side by side -- this is a legitimate ablation for the final report even if the answer is "none of the simple methods work well."